In [0]:
dbutils.widgets.removeAll()

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.text("input_table_name", input_table_list[0])

catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
input_table_name=dbutils.widgets.get("input_table_name")

df = spark.read.table(f"{catalog}.{schema}.{input_table_name}")
table_name = f"{catalog}.{schema}.{input_table_name}"
print(f"table_name:{table_name}")


In [0]:
 from pyspark.ml.feature import PCA
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.sql import functions as F

table_name = f"{catalog}.{schema}.{input_table_name}"
pca_column_name=[]
dims = [5, 10, 15, 20]

df = (
    spark.table(table_name)
    .withColumn("features", array_to_vector("text_embedding"))
)

result = df

for k in dims:
    pca = PCA(k=k, inputCol="features", outputCol=f"pca_vec_{k}")
    model = pca.fit(df)
    col_name=f"text_embedding_pca_{k}"
    projected = model.transform(df).select(
        "id",
        vector_to_array(F.col(f"pca_vec_{k}"), "float32").alias(col_name)
    )
    pca_column_name.append(col_name)
    result = result.join(projected, on="id", how="left")

result = result.drop("features")
display(result)

In [0]:
result.select(["id"]+ pca_column_name).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{table_name}_pca_dimensions")

In [0]:
df